# Rheology Workbook Data Audit

This notebook performs a reproducible, read-only quality-control audit of the
source Excel workbooks containing bovine-collagen hydrogel rheology measurements.

<details>
<summary><strong>Notebook purpose, scope, outputs and interpretation constraints</strong></summary>
<br>

## Objective

The objective of this notebook is to evaluate the structure, consistency,
traceability and analytical readiness of the source rheology workbooks before
schema standardisation, dataset integration or exploratory analysis.

## Audit scope

The workflow:

- verifies the integrity and provenance of the downloaded source archive;
- inventories all extracted Excel workbooks;
- parses concentration, replicate and test-type metadata from the source filenames;
- records workbook names, worksheet names, dimensions and populated ranges;
- compares workbook headings and unit labels against working expected structures;
- identifies missing, unexpected or undocumented cells and columns;
- inspects anomalous unit labels at the Unicode-character level;
- validates approved unit-label mappings;
- verifies unusual rheological values without modifying them; and
- exports machine-readable quality-control records for subsequent processing.

A total of **20 rheology workbooks** are included:

- **10 frequency-sweep workbooks**;
- **10 time-sweep workbooks**; and
- **660 measurement rows** across both test types.

## Working-schema terminology

The heading and unit patterns used in this audit are **working expected
schemas inferred from the repeated workbook structure and available source
metadata**.

They are used to identify structural inconsistencies during quality control.
They should not be interpreted as the final canonical data schema.

The formal canonical schema, including standardized field names, data types,
units, required fields, missing-value rules and validation constraints, will be
defined separately during the schema-standardisation stage.

## Important time-sweep limitation

The workbooks identified by the source filenames as time sweeps do not contain
an explicit elapsed-time column.

The `Meas. Pts.` field therefore represents measurement-point order only. It
must not be interpreted or labelled as elapsed time in seconds or minutes
unless the measurement interval is confirmed from instrument settings,
experimental documentation or other authoritative source metadata.

Until that information is available, analyses of these workbooks will use
**measurement point** as the horizontal coordinate.

## Confirmed audit findings

The audit documented **eight quality-control findings**:

| Finding category | Number of findings |
|---|---:|
| Unexpected schema column | 1 |
| Zero storage-modulus observation | 1 |
| Non-standard Unicode unit labels | 6 |
| **Total** | **8** |

These findings have documented and reproducible processing decisions. A
resolution status indicates that a data-handling rule has been established; it
does not necessarily mean that the underlying experimental cause is known.

## Data-handling principles

- Original Excel workbooks are treated as immutable source records.
- No numerical rheological measurement is silently corrected or deleted.
- Original labels and cell locations are retained for provenance.
- Recognised unit-label variants are standardised only in processed datasets.
- Uncertain numerical observations are retained with explicit quality-control flags.
- Undocumented fields are excluded from standardised datasets only through documented rules.
- Every processing decision remains traceable to its source workbook,
  worksheet and cell or cell range.

## Main outputs

This notebook creates the following reusable metadata and quality-control files:

```text
metadata/source_manifest.csv
metadata/extracted_file_inventory.csv
metadata/workbook_structure_inventory.csv
metadata/workbook_schema_qc.csv
data/quality_control/unit_label_mapping_audit.csv
data/quality_control/data_quality_issue_log.csv
```

These files will support canonical schema definition, controlled data extraction,
standardisation, validation and subsequent exploratory data analysis.

</details>


In [ ]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import os
import sys
import platform

# Identify the active Python environment and project folder
working_folder = Path.cwd().resolve()
project_root = (
    working_folder.parent
    if working_folder.name.lower() in {"notebook", "notebooks"}
    else working_folder
)

print("=" * 70)
print("PYTHON ENVIRONMENT")
print("=" * 70)
print(f"Python version    : {sys.version.split()[0]}")
print(f"Conda environment : {os.environ.get('CONDA_DEFAULT_ENV', 'Not detected')}")
print(f"Operating system  : {platform.platform()}")
print(f"Notebook folder   : {working_folder.name}")
print(f"Project folder    : {project_root.name}")

# Check packages that may be needed later
packages_to_check = [
    "pandas",
    "numpy",
    "openpyxl",
    "matplotlib",
    "seaborn",
    "scipy",
    "pytest",
]

print("\n" + "=" * 70)
print("RELEVANT PACKAGE VERSIONS")
print("=" * 70)

for package_name in packages_to_check:
    try:
        package_version = version(package_name)
        print(f"{package_name:<15}: {package_version}")
    except PackageNotFoundError:
        print(f"{package_name:<15}: NOT INSTALLED")

# List only the immediate contents of the current folder
# This does not open, extract, rename or modify any file.
visible_items = sorted(
    [item for item in working_folder.iterdir() if not item.name.startswith(".")],
    key=lambda item: (not item.is_dir(), item.name.lower())
)

print("\n" + "=" * 70)
print("ITEMS IN THE CURRENT WORKING FOLDER")
print("=" * 70)

if not visible_items:
    print("The current working folder is empty.")
else:
    for item in visible_items:
        if item.is_file():
            print(
                f"FILE   | {item.name} | "
                f"{item.stat().st_size:,} bytes | "
                f"type: {item.suffix.lower() or 'no extension'}"
            )
        elif item.is_dir():
            print(f"FOLDER | {item.name}")
        else:
            print(f"OTHER  | {item.name}")

In [ ]:
from pathlib import Path
import hashlib

# Locate the project root and original archive without modifying it
current_directory = Path.cwd().resolve()
project_root = (
    current_directory.parent
    if current_directory.name.lower() in {"notebook", "notebooks"}
    else current_directory
)
raw_file = project_root / "data" / "raw" / "Reología.rar"

# MD5 value published on the official Zenodo record
expected_md5 = "6ff4b0c9b58d8c470c5cd39d48558027"

def calculate_checksum(file_path, algorithm, chunk_size=1024 * 1024):
    """Calculate a checksum by reading the file in small chunks."""
    hash_object = hashlib.new(algorithm)

    with file_path.open("rb") as file:
        while chunk := file.read(chunk_size):
            hash_object.update(chunk)

    return hash_object.hexdigest()

print("=" * 70)
print("RAW ARCHIVE INTEGRITY CHECK")
print("=" * 70)
print(f"File exists       : {raw_file.exists()}")

if raw_file.exists():
    calculated_md5 = calculate_checksum(raw_file, "md5")
    calculated_sha256 = calculate_checksum(raw_file, "sha256")

    print(f"File name         : {raw_file.name}")
    print(f"File location     : {raw_file.relative_to(project_root).as_posix()}")
    print(f"File size         : {raw_file.stat().st_size:,} bytes")
    print(f"Calculated MD5    : {calculated_md5}")
    print(f"Expected MD5      : {expected_md5}")
    print(f"MD5 result        : {'MATCH' if calculated_md5 == expected_md5 else 'DOES NOT MATCH'}")
    print(f"Calculated SHA256 : {calculated_sha256}")
else:
    print("STOP: The archive was not found at the expected location.")

In [ ]:
from pathlib import Path
import hashlib
import csv

# Define project paths
current_directory = Path.cwd().resolve()
project_root = (
    current_directory.parent
    if current_directory.name.lower() in {"notebook", "notebooks"}
    else current_directory
)
raw_file = project_root / "data" / "raw" / "Reología.rar"
metadata_folder = project_root / "metadata"
manifest_file = metadata_folder / "source_manifest.csv"

# Confirm that the original archive exists
if not raw_file.is_file():
    raise FileNotFoundError(
        f"The raw archive was not found here:\n{raw_file.resolve()}"
    )

# Create the metadata folder without changing the raw-data folder
metadata_folder.mkdir(parents=True, exist_ok=True)

# Read the archive only for checksum calculation
raw_file_bytes = raw_file.read_bytes()

# Calculate digital fingerprints
calculated_md5 = hashlib.md5(raw_file_bytes).hexdigest()
calculated_sha256 = hashlib.sha256(raw_file_bytes).hexdigest()

# MD5 published on the official Zenodo record
expected_md5 = "6ff4b0c9b58d8c470c5cd39d48558027"

# Stop if the archive does not match the official Zenodo file
if calculated_md5 != expected_md5:
    raise ValueError(
        "The archive does not match the MD5 published on Zenodo. "
        "Do not extract or analyse it."
    )

# Create the source-provenance record
source_record = {
    "dataset_id": "zenodo_17413651",
    "dataset_title": (
        "PID2020-113790RB-I00 data set: "
        "hydrogel viscoelastic properties from rheology tests"
    ),
    "repository": "Zenodo",
    "record_version": "v1",
    "dataset_doi": "10.5281/zenodo.17413651",
    "dataset_url": "https://zenodo.org/records/17413651",
    "publication_date": "2025-10-22",
    "related_paper_doi": "10.1016/j.actbio.2024.07.002",
    # Original download date confirmed from browser download history.
    "retrieval_date": "2026-09-15",
    "original_filename": raw_file.name,
    "file_size_bytes": raw_file.stat().st_size,
    "md5": calculated_md5,
    "sha256": calculated_sha256,
    "access_status": "Publicly accessible",
    "licence_status": (
        "No specific licence displayed on the Zenodo record; "
        "redistribution not yet confirmed"
    ),
    "material_system": "Collagen hydrogel",
    "collagen_concentrations_mg_ml": "0.8; 1.5; 2.3",
    "reported_temperature_c": 37,
    "reported_test_types": "Time sweep; frequency sweep",
    "reported_frequency_range_hz": "0.1-10",
    "reported_strain_amplitude_percent": 1,
    "reported_geometry": "25 mm standard steel cone",
    "metadata_source": "Zenodo record description",
}

# Write the provenance record using Python's standard CSV module
with manifest_file.open(
    mode="w",
    newline="",
    encoding="utf-8-sig"
) as csv_file:
    writer = csv.DictWriter(
        csv_file,
        fieldnames=source_record.keys()
    )
    writer.writeheader()
    writer.writerow(source_record)

# Confirm that the file was created successfully
print("=" * 70)
print("SOURCE PROVENANCE MANIFEST")
print("=" * 70)
print(f"Raw archive found : {raw_file.is_file()}")
print(f"MD5 verified      : {calculated_md5 == expected_md5}")
print(f"Manifest created  : {manifest_file.is_file()}")
print("Number of records : 1")
print(f"Saved location    : {manifest_file.relative_to(project_root).as_posix()}")

print("\n" + "=" * 70)
print("RECORDED METADATA")
print("=" * 70)

for field_name, recorded_value in source_record.items():
    print(f"{field_name:<40}: {recorded_value}")

In [ ]:
import sys
import numpy as np
import pandas as pd
import openpyxl
import matplotlib
import seaborn as sns
import scipy

print("=" * 70)
print("HYDROGEL-RHEOLOGY ENVIRONMENT CHECK")
print("=" * 70)

print(f"Python version     : {sys.version.split()[0]}")
print(f"NumPy version      : {np.__version__}")
print(f"pandas version     : {pd.__version__}")
print(f"openpyxl version   : {openpyxl.__version__}")
print(f"Matplotlib version : {matplotlib.__version__}")
print(f"Seaborn version    : {sns.__version__}")
print(f"SciPy version      : {scipy.__version__}")

print("=" * 70)

if np.__version__.startswith("1.26"):
    print("ENVIRONMENT STATUS : PASS")
else:
    print("ENVIRONMENT STATUS : CHECK NUMPY VERSION")

In [ ]:
from pathlib import Path

current_directory = Path.cwd().resolve()
project_root = (
    current_directory.parent
    if current_directory.name.lower() in {"notebook", "notebooks"}
    else current_directory
)
environment_file = project_root / "environment.yml"

environment_text = """name: hydrogel-rheology

channels:
  - defaults

dependencies:
  - python=3.11
  - numpy=1.26
  - pandas=2.2
  - openpyxl
  - matplotlib
  - seaborn
  - scipy
  - notebook
  - jupyterlab
  - ipykernel
  - pytest
"""

environment_file.write_text(environment_text, encoding="utf-8")

print("=" * 70)
print("ENVIRONMENT FILE CREATED")
print("=" * 70)
print(f"File name     : {environment_file.name}")
print(f"Saved location: {environment_file.relative_to(project_root).as_posix()}")
print("\nFILE CONTENTS")
print("-" * 70)
print(environment_file.read_text(encoding="utf-8"))

In [ ]:
from pathlib import Path
from IPython.display import display
import hashlib
import re
import pandas as pd

# ------------------------------------------------------------
# 1. Define the source and output locations
# ------------------------------------------------------------
current_directory = Path.cwd().resolve()
project_root = (
    current_directory.parent
    if current_directory.name.lower() in {"notebook", "notebooks"}
    else current_directory
)

extracted_folder = (
    project_root
    / "data"
    / "interim"
    / "zenodo_17413651_v1"
)

inventory_file = (
    project_root
    / "metadata"
    / "extracted_file_inventory.csv"
)

# ------------------------------------------------------------
# 2. Define how information is encoded in each filename
# Example:
# Colageno_bov_1.5_2_frecuencia.xlsx
# ------------------------------------------------------------
filename_pattern = re.compile(
    r"^Col[aá]geno_bov_"
    r"(?P<concentration>\d+(?:\.\d+)?)_"
    r"(?P<replicate>\d+)_"
    r"(?P<test_type>frecuencia|tiempo)$",
    flags=re.IGNORECASE
)

test_type_mapping = {
    "frecuencia": "frequency_sweep",
    "tiempo": "time_sweep",
}

# ------------------------------------------------------------
# 3. Inventory every extracted file
# ------------------------------------------------------------
records = []

for file_path in sorted(extracted_folder.rglob("*")):
    if not file_path.is_file():
        continue

    match = filename_pattern.fullmatch(file_path.stem)

    sha256_hash = hashlib.sha256()
    with file_path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(8192), b""):
            sha256_hash.update(chunk)

    if match:
        concentration = float(match.group("concentration"))
        replicate = int(match.group("replicate"))
        original_test_type = match.group("test_type").lower()
        standard_test_type = test_type_mapping[original_test_type]
        filename_parsed = True
    else:
        concentration = None
        replicate = None
        original_test_type = None
        standard_test_type = None
        filename_parsed = False

    records.append(
        {
            "dataset_id": "zenodo_17413651",
            "original_filename": file_path.name,
            "relative_path": file_path.relative_to(project_root).as_posix(),
            "extension": file_path.suffix.lower(),
            "file_size_bytes": file_path.stat().st_size,
            "sha256": sha256_hash.hexdigest(),
            "collagen_concentration_mg_ml": concentration,
            "replicate_id": replicate,
            "test_type_original": original_test_type,
            "test_type_standard": standard_test_type,
            "metadata_source": "filename_parsing",
            "filename_parsed": filename_parsed,
        }
    )

inventory = pd.DataFrame(records)

if not inventory.empty:
    inventory["replicate_id"] = inventory["replicate_id"].astype("Int64")

inventory.to_csv(inventory_file, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 4. Display the audit results
# ------------------------------------------------------------
print("=" * 80)
print("EXTRACTED RHEOLOGY FILE INVENTORY")
print("=" * 80)
print(f"Extracted folder       : {extracted_folder.relative_to(project_root).as_posix()}")
print(f"Files found            : {len(inventory)}")
print(f"Filenames parsed       : {int(inventory['filename_parsed'].sum())}")
print(f"Unparsed filenames     : {int((~inventory['filename_parsed']).sum())}")
print(f"Inventory saved to     : {inventory_file.relative_to(project_root).as_posix()}")

print("\nFILES BY CONCENTRATION AND TEST TYPE")
print("-" * 80)

summary = pd.crosstab(
    inventory["collagen_concentration_mg_ml"],
    inventory["test_type_standard"],
    margins=True,
)

display(summary)

print("\nFILE-LEVEL METADATA")
print("-" * 80)

display(
    inventory[
        [
            "original_filename",
            "collagen_concentration_mg_ml",
            "replicate_id",
            "test_type_standard",
            "extension",
            "file_size_bytes",
            "filename_parsed",
        ]
    ]
)

In [ ]:
from pathlib import Path
from IPython.display import display
from openpyxl import load_workbook
import pandas as pd

# ------------------------------------------------------------
# 1. Load the file inventory
# ------------------------------------------------------------
current_directory = Path.cwd().resolve()
project_root = (
    current_directory.parent
    if current_directory.name.lower() in {"notebook", "notebooks"}
    else current_directory
)

inventory_file = (
    project_root
    / "metadata"
    / "extracted_file_inventory.csv"
)

structure_file = (
    project_root
    / "metadata"
    / "workbook_structure_inventory.csv"
)

inventory = pd.read_csv(inventory_file)

# ------------------------------------------------------------
# 2. Record the worksheets and dimensions of all workbooks
# ------------------------------------------------------------
structure_records = []

for record in inventory.itertuples(index=False):
    workbook_path = project_root / Path(record.relative_path)

    workbook = load_workbook(
        workbook_path,
        read_only=True,
        data_only=False
    )

    for sheet_name in workbook.sheetnames:
        worksheet = workbook[sheet_name]

        structure_records.append(
            {
                "original_filename": record.original_filename,
                "collagen_concentration_mg_ml":
                    record.collagen_concentration_mg_ml,
                "replicate_id": record.replicate_id,
                "test_type_standard": record.test_type_standard,
                "sheet_name": sheet_name,
                "sheet_state": worksheet.sheet_state,
                "reported_cell_range": worksheet.calculate_dimension(),
                "reported_max_rows": worksheet.max_row,
                "reported_max_columns": worksheet.max_column,
            }
        )

    workbook.close()

structure_inventory = pd.DataFrame(structure_records)

structure_inventory.to_csv(
    structure_file,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 80)
print("EXCEL WORKBOOK STRUCTURE AUDIT")
print("=" * 80)
print(f"Workbooks checked       : {len(inventory)}")
print(f"Worksheets found        : {len(structure_inventory)}")
print(f"Structure report saved  : {structure_file.relative_to(project_root).as_posix()}")

display(structure_inventory)

# ------------------------------------------------------------
# 3. Select one representative frequency sweep and time sweep
# ------------------------------------------------------------
representative_files = (
    inventory
    .sort_values(
        [
            "collagen_concentration_mg_ml",
            "replicate_id",
            "test_type_standard",
        ]
    )
    .drop_duplicates(subset="test_type_standard")
)

# ------------------------------------------------------------
# 4. Preview the first 30 Excel rows without changing anything
# ------------------------------------------------------------
for _, record in representative_files.iterrows():
    workbook_path = project_root / Path(record["relative_path"])

    workbook = load_workbook(
        workbook_path,
        read_only=True,
        data_only=False
    )

    print("\n" + "=" * 80)
    print(f"REPRESENTATIVE FILE : {record['original_filename']}")
    print(f"TEST TYPE           : {record['test_type_standard']}")
    print("=" * 80)

    for sheet_name in workbook.sheetnames:
        worksheet = workbook[sheet_name]

        preview_rows = list(
            worksheet.iter_rows(
                min_row=1,
                max_row=min(worksheet.max_row, 30),
                values_only=True
            )
        )

        preview = pd.DataFrame(preview_rows)
        preview = preview.dropna(axis=0, how="all")
        preview = preview.dropna(axis=1, how="all")

        print(f"\nWorksheet           : {sheet_name}")
        print(f"Reported dimensions : {worksheet.calculate_dimension()}")

        if preview.empty:
            print("No populated cells found in the first 30 rows.")
        else:
            preview.index = preview.index + 1
            preview.index.name = "Excel row"
            preview.columns = [
                f"Excel column {column_number + 1}"
                for column_number in preview.columns
            ]
            display(preview)

    workbook.close()

In [ ]:
from pathlib import Path
from openpyxl import load_workbook
import pandas as pd

current_directory = Path.cwd().resolve()
project_root = (
    current_directory.parent
    if current_directory.name.lower() in {"notebook", "notebooks"}
    else current_directory
)

inventory = pd.read_csv(
    project_root
    / "metadata"
    / "extracted_file_inventory.csv"
)

# Select one representative file for each test type
representative_files = (
    inventory
    .sort_values(
        [
            "collagen_concentration_mg_ml",
            "replicate_id",
            "test_type_standard",
        ]
    )
    .drop_duplicates(subset="test_type_standard")
    .sort_values("test_type_standard")
)

print("=" * 80)
print("COMPACT RHEOLOGY WORKBOOK PREVIEW")
print("=" * 80)

for _, record in representative_files.iterrows():
    workbook_path = project_root / Path(record["relative_path"])

    workbook = load_workbook(
        workbook_path,
        read_only=True,
        data_only=False
    )

    worksheet = workbook[workbook.sheetnames[0]]

    print("\n" + "=" * 80)
    print(f"FILE      : {record['original_filename']}")
    print(f"TEST TYPE : {record['test_type_standard']}")
    print(f"WORKSHEET : {worksheet.title}")
    print(f"RANGE     : {worksheet.calculate_dimension()}")
    print("-" * 80)

    # Print only Excel rows 4 to 10
    for excel_row_number, row_values in enumerate(
        worksheet.iter_rows(
            min_row=4,
            max_row=min(worksheet.max_row, 10),
            values_only=True
        ),
        start=4
    ):
        print(f"Excel row {excel_row_number}: {row_values}")

    workbook.close()

In [ ]:
from pathlib import Path
from IPython.display import display
from openpyxl import load_workbook
import pandas as pd

current_directory = Path.cwd().resolve()
project_root = (
    current_directory.parent
    if current_directory.name.lower() in {"notebook", "notebooks"}
    else current_directory
)

inventory = pd.read_csv(
    project_root
    / "metadata"
    / "extracted_file_inventory.csv"
)

qc_file = (
    project_root
    / "metadata"
    / "workbook_schema_qc.csv"
)

# Expected schemas learned from the representative workbooks
expected_schemas = {
    "frequency_sweep": {
        "headers": (
            "Meas. Pts.",
            "Angular Frequency",
            "Storage Modulus",
            "Loss Modulus",
            "Temperature",
        ),
        "units": (
            None,
            "[rad/s]",
            "[Pa]",
            "[Pa]",
            "[°C]",
        ),
    },
    "time_sweep": {
        "headers": (
            "Meas. Pts.",
            "Shear Stress",
            "Angular Frequency",
            "Storage Modulus",
            "Strain",
            "Complex Viscosity",
            "Loss Modulus",
            "Temperature",
        ),
        "units": (
            None,
            "[Pa]",
            "[rad/s]",
            "[Pa]",
            "[%]",
            "[Pa·s]",
            "[Pa]",
            "[°C]",
        ),
    },
}

qc_records = []

for record in inventory.itertuples(index=False):
    workbook_path = project_root / Path(record.relative_path)

    workbook = load_workbook(
        workbook_path,
        read_only=True,
        data_only=False
    )

    worksheet = workbook[workbook.sheetnames[0]]

    headers = tuple(
        next(
            worksheet.iter_rows(
                min_row=4,
                max_row=4,
                values_only=True
            )
        )
    )

    units = tuple(
        next(
            worksheet.iter_rows(
                min_row=5,
                max_row=5,
                values_only=True
            )
        )
    )

    raw_data_rows = list(
        worksheet.iter_rows(
            min_row=6,
            max_row=worksheet.max_row,
            values_only=True
        )
    )

    # Remove completely empty Excel rows
    data_rows = [
        row for row in raw_data_rows
        if any(value is not None for value in row)
    ]

    data = pd.DataFrame(data_rows, columns=headers)
    numeric_data = data.apply(pd.to_numeric, errors="coerce")

    non_numeric_count = int(
        (data.notna() & numeric_data.isna()).sum().sum()
    )

    missing_value_count = int(data.isna().sum().sum())

    storage_zero_count = int(
        (numeric_data["Storage Modulus"] == 0).sum()
    )

    loss_zero_count = int(
        (numeric_data["Loss Modulus"] == 0).sum()
    )

    expected = expected_schemas[record.test_type_standard]

    qc_records.append(
        {
            "original_filename": record.original_filename,
            "collagen_concentration_mg_ml":
                record.collagen_concentration_mg_ml,
            "replicate_id": record.replicate_id,
            "test_type_standard": record.test_type_standard,
            "data_row_count": len(data),
            "headers_match_expected":
                headers == expected["headers"],
            "units_match_expected":
                units == expected["units"],
            "missing_value_count": missing_value_count,
            "non_numeric_value_count": non_numeric_count,
            "storage_modulus_zero_count": storage_zero_count,
            "loss_modulus_zero_count": loss_zero_count,
            "headers_found": " | ".join(
                "" if value is None else str(value)
                for value in headers
            ),
            "units_found": " | ".join(
                "" if value is None else str(value)
                for value in units
            ),
        }
    )

    workbook.close()

qc_results = pd.DataFrame(qc_records)

# Determine the usual number of rows for each test type
usual_row_count = (
    qc_results
    .groupby("test_type_standard")["data_row_count"]
    .transform(lambda values: values.mode().iloc[0])
)

qc_results["row_count_matches_mode"] = (
    qc_results["data_row_count"] == usual_row_count
)

qc_results.to_csv(
    qc_file,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 80)
print("WORKBOOK SCHEMA AND BASIC DATA QC")
print("=" * 80)
print(f"Files checked            : {len(qc_results)}")
print(
    "Correct header schemas   : "
    f"{int(qc_results['headers_match_expected'].sum())}"
    f"/{len(qc_results)}"
)
print(
    "Correct unit schemas     : "
    f"{int(qc_results['units_match_expected'].sum())}"
    f"/{len(qc_results)}"
)
print(f"QC report saved          : {qc_file.relative_to(project_root).as_posix()}")

print("\nDATA-ROW COUNT SUMMARY")
print("-" * 80)

row_summary = (
    qc_results
    .groupby(
        ["test_type_standard", "data_row_count"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "number_of_files"})
)

display(row_summary)

issue_mask = (
    ~qc_results["headers_match_expected"]
    | ~qc_results["units_match_expected"]
    | ~qc_results["row_count_matches_mode"]
    | (qc_results["missing_value_count"] > 0)
    | (qc_results["non_numeric_value_count"] > 0)
)

print("\nSCHEMA OR DATA-COMPLETENESS ISSUES")
print("-" * 80)

if not issue_mask.any():
    print("None detected.")
else:
    display(
        qc_results.loc[
            issue_mask,
            [
                "original_filename",
                "test_type_standard",
                "data_row_count",
                "headers_match_expected",
                "units_match_expected",
                "row_count_matches_mode",
                "missing_value_count",
                "non_numeric_value_count",
            ],
        ]
    )

print("\nZERO-MODULUS VALUES TO REVIEW")
print("-" * 80)

zero_flags = qc_results.loc[
    (qc_results["storage_modulus_zero_count"] > 0)
    | (qc_results["loss_modulus_zero_count"] > 0),
    [
        "original_filename",
        "storage_modulus_zero_count",
        "loss_modulus_zero_count",
    ],
]

if zero_flags.empty:
    print("No zero modulus values detected.")
else:
    display(zero_flags)

In [ ]:
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
import pandas as pd

# ------------------------------------------------------------
# 1. Load the existing audit files
# ------------------------------------------------------------
current_directory = Path.cwd().resolve()
project_root = (
    current_directory.parent
    if current_directory.name.lower() in {"notebook", "notebooks"}
    else current_directory
)

inventory = pd.read_csv(
    project_root
    / "metadata"
    / "extracted_file_inventory.csv"
)

qc_results = pd.read_csv(
    project_root
    / "metadata"
    / "workbook_schema_qc.csv"
)

file_path_lookup = dict(
    zip(
        inventory["original_filename"],
        inventory["relative_path"]
    )
)

# ------------------------------------------------------------
# 2. Define the expected schemas
# ------------------------------------------------------------
expected_schemas = {
    "frequency_sweep": {
        "headers": (
            "Meas. Pts.",
            "Angular Frequency",
            "Storage Modulus",
            "Loss Modulus",
            "Temperature",
        ),
        "units": (
            None,
            "[rad/s]",
            "[Pa]",
            "[Pa]",
            "[°C]",
        ),
    },
    "time_sweep": {
        "headers": (
            "Meas. Pts.",
            "Shear Stress",
            "Angular Frequency",
            "Storage Modulus",
            "Strain",
            "Complex Viscosity",
            "Loss Modulus",
            "Temperature",
        ),
        "units": (
            None,
            "[Pa]",
            "[rad/s]",
            "[Pa]",
            "[%]",
            "[Pa·s]",
            "[Pa]",
            "[°C]",
        ),
    },
}

# Convert saved True/False values safely
header_match = (
    qc_results["headers_match_expected"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

unit_match = (
    qc_results["units_match_expected"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

problem_files = qc_results.loc[
    (~header_match)
    | (~unit_match)
    | (qc_results["missing_value_count"] > 0)
].copy()

# ------------------------------------------------------------
# 3. Inspect every workbook with a detected difference
# ------------------------------------------------------------
print("=" * 90)
print("DETAILED SCHEMA DIFFERENCE CHECK")
print("=" * 90)
print(f"Files requiring inspection: {len(problem_files)}")

for _, record in problem_files.iterrows():
    workbook_path = (
        project_root
        / Path(file_path_lookup[record["original_filename"]])
    )

    workbook = load_workbook(
        workbook_path,
        read_only=True,
        data_only=False
    )

    worksheet = workbook[workbook.sheetnames[0]]

    headers_found = tuple(
        next(
            worksheet.iter_rows(
                min_row=4,
                max_row=4,
                values_only=True
            )
        )
    )

    units_found = tuple(
        next(
            worksheet.iter_rows(
                min_row=5,
                max_row=5,
                values_only=True
            )
        )
    )

    expected = expected_schemas[
        record["test_type_standard"]
    ]

    # Calculate cell coordinates manually.
    # This avoids the EmptyCell.coordinate error.
    missing_cells = []

    for excel_row_number, row_values in enumerate(
        worksheet.iter_rows(
            min_row=6,
            max_row=worksheet.max_row,
            values_only=True
        ),
        start=6
    ):
        for column_number, value in enumerate(
            row_values,
            start=1
        ):
            if value is None:
                cell_address = (
                    f"{get_column_letter(column_number)}"
                    f"{excel_row_number}"
                )
                missing_cells.append(cell_address)

    print("\n" + "-" * 90)
    print(f"FILE             : {record['original_filename']}")
    print(f"TEST TYPE        : {record['test_type_standard']}")
    print(f"EXPECTED HEADERS : {expected['headers']}")
    print(f"HEADERS FOUND    : {headers_found}")
    print(f"EXPECTED UNITS   : {expected['units']}")
    print(f"UNITS FOUND      : {units_found}")

    if missing_cells:
        print(f"MISSING CELLS    : {missing_cells}")
    else:
        print("MISSING CELLS    : None")

    workbook.close()

# ------------------------------------------------------------
# 4. Print every file containing zero modulus readings
# ------------------------------------------------------------
print("\n" + "=" * 90)
print("FILES CONTAINING ZERO MODULUS VALUES")
print("=" * 90)

zero_files = qc_results.loc[
    (qc_results["storage_modulus_zero_count"] > 0)
    | (qc_results["loss_modulus_zero_count"] > 0),
    [
        "original_filename",
        "storage_modulus_zero_count",
        "loss_modulus_zero_count",
    ],
]

if zero_files.empty:
    print("No zero modulus readings found.")
else:
    print(zero_files.to_string(index=False))

<details>
<summary><strong>Workbook schema and data-quality audit</strong></summary>
<br>

<h3>Audit scope</h3>

<p>
A reproducible quality-control audit was performed on all 20 extracted
rheology workbooks before data harmonization, integration, or scientific
analysis.
</p>

<ul>
  <li><strong>Frequency-sweep workbooks:</strong> 10</li>
  <li><strong>Measurement points per frequency sweep:</strong> 21</li>
  <li><strong>Time-sweep workbooks:</strong> 10</li>
  <li><strong>Measurement points per time sweep:</strong> 45</li>
  <li><strong>Workbooks matching the expected header schema:</strong> 19 of 20</li>
  <li><strong>Workbooks matching the expected unit schema:</strong> 15 of 20</li>
  <li><strong>Workbooks requiring additional inspection:</strong> 5</li>
</ul>

<p>
A workbook was flagged when its column headings, unit labels, or populated-cell
structure differed from the expected schema. A quality-control flag identifies
a condition requiring review; it does not, by itself, indicate that the workbook
or its measurements are unusable.
</p>

<h3>Audit findings</h3>

<h4>1. Unexpected unnamed column</h4>

<p>
The frequency-sweep workbook
<code>Colageno_bov_0.8_2_frecuencia.xlsx</code> contains an unexpected sixth
column, column F, without a heading or unit label. Within the audited range,
cells <code>F6:F15</code> are blank.
</p>

<p>
The five expected frequency-sweep variables remain present. However, because
column F falls outside the expected schema, any remaining populated cells in
this column must be inspected before the column can be interpreted, mapped, or
excluded. No raw column will be removed without documented justification.
</p>

<h4>2. Non-standard unit-label characters</h4>

<p>
Unit-label differences were identified in the following four workbooks:
</p>

<table>
  <thead>
    <tr>
      <th>Workbook</th>
      <th>Affected unit label</th>
      <th>Expected canonical form</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>Colageno_bov_1.5_1_frecuencia.xlsx</code></td>
      <td>Temperature</td>
      <td><code>[°C]</code></td>
    </tr>
    <tr>
      <td><code>Colageno_bov_1.5_1_tiempo.xlsx</code></td>
      <td>Complex viscosity and temperature</td>
      <td><code>[Pa·s]</code> and <code>[°C]</code></td>
    </tr>
    <tr>
      <td><code>Colageno_bov_2.3_2_frecuencia.xlsx</code></td>
      <td>Temperature</td>
      <td><code>[°C]</code></td>
    </tr>
    <tr>
      <td><code>Colageno_bov_2.3_2_tiempo.xlsx</code></td>
      <td>Complex viscosity and temperature</td>
      <td><code>[Pa·s]</code> and <code>[°C]</code></td>
    </tr>
  </tbody>
</table>

<p>
The expected variable headings and measurement cells are otherwise present in
these workbooks. The observed differences are consistent with character-encoding
or instrument-export inconsistencies; they do not provide evidence of missing
experimental measurements.
</p>

<p>
The original unit labels will be retained in the raw-data record. Documented
mapping rules will convert recognized variants to the canonical forms
<code>[Pa·s]</code> and <code>[°C]</code> in the processed dataset.
</p>

<h4>3. Zero storage-modulus observation</h4>

<p>
One zero storage-modulus value was detected in
<code>Colageno_bov_0.8_1_frecuencia.xlsx</code>.
</p>

<table>
  <thead>
    <tr>
      <th>Parameter</th>
      <th>Observed value</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Measurement point</td>
      <td>1</td>
    </tr>
    <tr>
      <td>Angular frequency, ω</td>
      <td>0.628 rad/s</td>
    </tr>
    <tr>
      <td>Equivalent frequency, f</td>
      <td>Approximately 0.100 Hz</td>
    </tr>
    <tr>
      <td>Storage modulus, G′</td>
      <td>0 Pa</td>
    </tr>
    <tr>
      <td>Loss modulus, G″</td>
      <td>1.59 Pa</td>
    </tr>
    <tr>
      <td>Temperature</td>
      <td>37 °C</td>
    </tr>
  </tbody>
</table>

<p>
No zero loss-modulus values were detected.
</p>

<p>
The zero value for G′ will be preserved and flagged rather than automatically
removed, replaced, or interpreted as missing. It could represent a measurement
below the instrument's reliable resolution, an export artifact, or a genuine
low-frequency response. The available evidence is insufficient to determine
its cause.
</p>

<p>
Because the loss tangent is calculated as
<code>tan δ = G″/G′</code>, it is undefined for this observation because
<code>G′ = 0</code>. Therefore, the derived loss-tangent value will be recorded
as missing or not calculated, while the original measurement row will remain
in the dataset.
</p>

<h3>Data-handling decision</h3>

<p>
The original workbooks will remain unchanged. Schema standardization, unit-label
mapping, validation flags, and derived calculations will be implemented
reproducibly in the processed-data workflow. Every correction, exclusion, and
unresolved issue will be documented with its source location, rationale, and
effect on subsequent analysis. No raw measurement will be silently altered or
deleted.
</p>

</details>

In [ ]:
# Step 5G — Read-only verification of flagged workbook observations

from pathlib import Path
from numbers import Number

import pandas as pd
from IPython.display import display
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter


# Search from the project root so this notebook works inside notebooks/
current_directory = Path.cwd().resolve()
project_root = (
    current_directory.parent
    if current_directory.name.lower() in {"notebook", "notebooks"}
    else current_directory
)
SEARCH_ROOT = project_root


def is_blank(value):
    """Return True only for an empty cell or an empty text value."""
    return value is None or (
        isinstance(value, str) and value.strip() == ""
    )


def normalise_text(value):
    """Normalise spacing for reliable header matching."""
    if value is None:
        return ""
    return " ".join(str(value).split())


def locate_workbook(filename):
    """Locate exactly one workbook without modifying it."""
    matches = [
        path.resolve()
        for path in SEARCH_ROOT.rglob(filename)
        if path.is_file() and not path.name.startswith("~$")
    ]

    matches = sorted(matches, key=lambda path: str(path).lower())

    if not matches:
        raise FileNotFoundError(
            f"Could not locate {filename!r} below {SEARCH_ROOT}."
        )

    if len(matches) > 1:
        relative_matches = [
            str(path.relative_to(SEARCH_ROOT)) for path in matches
        ]
        raise RuntimeError(
            f"Multiple copies of {filename!r} were found:\n"
            + "\n".join(relative_matches)
        )

    return matches[0]


def find_sheet_and_header(workbook, expected_header):
    """Find the worksheet and cell containing a required header."""
    for worksheet in workbook.worksheets:
        rows_to_scan = min(worksheet.max_row, 20)

        for row_number in range(1, rows_to_scan + 1):
            for column_number in range(1, worksheet.max_column + 1):
                value = worksheet.cell(
                    row=row_number,
                    column=column_number
                ).value

                if normalise_text(value) == expected_header:
                    return worksheet, row_number, column_number

    raise ValueError(
        f"Header {expected_header!r} was not found in any worksheet."
    )


# ============================================================
# Check 1: Unexpected column F
# ============================================================

extra_column_filename = "Colageno_bov_0.8_2_frecuencia.xlsx"
extra_column_path = locate_workbook(extra_column_filename)

extra_workbook = load_workbook(
    extra_column_path,
    read_only=True,
    data_only=False
)

extra_sheet, header_row, _ = find_sheet_and_header(
    extra_workbook,
    "Angular Frequency"
)

data_start_row = header_row + 2

column_f_records = []

for row_number in range(1, extra_sheet.max_row + 1):
    column_f_value = extra_sheet.cell(
        row=row_number,
        column=6
    ).value

    if not is_blank(column_f_value):
        record = {"Excel row": row_number}

        # Include columns A–F so that any column-F value has context
        for column_number in range(1, 7):
            column_letter = get_column_letter(column_number)
            record[column_letter] = extra_sheet.cell(
                row=row_number,
                column=column_number
            ).value

        column_f_records.append(record)


print("=" * 72)
print("CHECK 1 — UNEXPECTED COLUMN F")
print("=" * 72)
print(f"Workbook: {extra_column_filename}")
print(f"Worksheet: {extra_sheet.title}")
print(
    f"Worksheet dimensions: "
    f"{extra_sheet.max_row} rows × {extra_sheet.max_column} columns"
)
print(
    f"Expected measurement-data range begins at Excel row "
    f"{data_start_row}."
)
print(
    f"Number of non-empty cells detected in column F: "
    f"{len(column_f_records)}"
)

if column_f_records:
    print("\nNon-empty column-F cells and their row context:")
    display(pd.DataFrame(column_f_records))
else:
    print(
        "\nColumn F contains no stored values. Its detection as an extra "
        "column may therefore be caused by formatting or workbook metadata."
    )

extra_workbook.close()


# ============================================================
# Check 2: Zero storage-modulus value
# ============================================================

zero_filename = "Colageno_bov_0.8_1_frecuencia.xlsx"
zero_path = locate_workbook(zero_filename)

# Load evaluated values and original cell contents separately
value_workbook = load_workbook(
    zero_path,
    read_only=True,
    data_only=True
)

raw_workbook = load_workbook(
    zero_path,
    read_only=True,
    data_only=False
)

value_sheet, zero_header_row, storage_column = find_sheet_and_header(
    value_workbook,
    "Storage Modulus"
)

raw_sheet = raw_workbook[value_sheet.title]
zero_data_start_row = zero_header_row + 2

zero_rows = []

for row_number in range(
    zero_data_start_row,
    value_sheet.max_row + 1
):
    storage_value = value_sheet.cell(
        row=row_number,
        column=storage_column
    ).value

    if (
        isinstance(storage_value, Number)
        and float(storage_value) == 0.0
    ):
        zero_rows.append(row_number)


print("\n" + "=" * 72)
print("CHECK 2 — ZERO STORAGE-MODULUS VALUE")
print("=" * 72)
print(f"Workbook: {zero_filename}")
print(f"Worksheet: {value_sheet.title}")
print(f"Storage Modulus column: {get_column_letter(storage_column)}")
print(f"Excel row(s) containing G′ = 0: {zero_rows}")


if zero_rows:
    # Display each zero row together with its neighbouring measurements
    rows_to_display = sorted({
        nearby_row
        for zero_row in zero_rows
        for nearby_row in (zero_row - 1, zero_row, zero_row + 1)
        if zero_data_start_row <= nearby_row <= value_sheet.max_row
    })

    column_labels = []

    for column_number in range(1, value_sheet.max_column + 1):
        header = normalise_text(
            value_sheet.cell(
                row=zero_header_row,
                column=column_number
            ).value
        )

        unit = normalise_text(
            value_sheet.cell(
                row=zero_header_row + 1,
                column=column_number
            ).value
        )

        if not header:
            header = f"Column {get_column_letter(column_number)}"

        label = f"{header} {unit}".strip()
        column_labels.append(label)

    nearby_records = []

    for row_number in rows_to_display:
        record = {"Excel row": row_number}

        for column_number, label in enumerate(
            column_labels,
            start=1
        ):
            record[label] = value_sheet.cell(
                row=row_number,
                column=column_number
            ).value

        nearby_records.append(record)

    print("\nZero observation and neighbouring measurement rows:")
    display(pd.DataFrame(nearby_records))

    raw_cell_details = []

    for row_number in zero_rows:
        raw_cell = raw_sheet.cell(
            row=row_number,
            column=storage_column
        )

        evaluated_cell = value_sheet.cell(
            row=row_number,
            column=storage_column
        )

        raw_cell_details.append({
            "Excel row": row_number,
            "Cell reference":
                f"{get_column_letter(storage_column)}{row_number}",
            "Stored cell content": raw_cell.value,
            "Evaluated value": evaluated_cell.value,
            "Cell data type": raw_cell.data_type
        })

    print("\nRaw storage-modulus cell details:")
    display(pd.DataFrame(raw_cell_details))

else:
    print(
        "\nNo numeric zero was found in the evaluated "
        "Storage Modulus column."
    )


value_workbook.close()
raw_workbook.close()

<details>
<summary><strong>Verified workbook observations and data-handling decisions</strong></summary>
<br>

<h3>Verification scope</h3>

<ul>
  <li>Two observations from the initial audit were checked directly in the original Excel workbooks.</li>
  <li>The verification was read-only.</li>
  <li>No original values, formulas, or worksheets were modified.</li>
</ul>

<h3>Observation 1: Undocumented formula column</h3>

<p>
<strong>Original file:</strong>
<code>Colageno_bov_0.8_2_frecuencia.xlsx</code><br>
<strong>Worksheet:</strong> <code>Hoja1</code>
</p>

<h4>Evidence</h4>

<ul>
  <li>The worksheet contains <strong>26 rows and 6 columns</strong>.</li>
  <li>The expected frequency-sweep schema contains only <strong>five columns, A–E</strong>.</li>
  <li>Column F does not have a heading or unit label.</li>
  <li>Cells <code>F6:F15</code> are blank.</li>
  <li>Cells <code>F16:F26</code> contain 11 Excel formulas.</li>
  <li>The formulas follow the pattern <code>=D[row]+1</code>.</li>
  <li>For example, cell <code>F16</code> contains <code>=D16+1</code>.</li>
  <li>Column D contains the loss modulus, G″.</li>
</ul>

<h4>Interpretation</h4>

<ul>
  <li>Column F numerically adds 1 to the corresponding loss-modulus value.</li>
  <li>The formulas are not part of the expected frequency-sweep schema.</li>
  <li>No heading, unit, or metadata explains the scientific purpose of this calculation.</li>
  <li>Column F is therefore classified as an <strong>undocumented derived column</strong>.</li>
  <li>Its intended meaning cannot be determined from the workbook alone.</li>
</ul>

<h4>Data-handling decision</h4>

<ul>
  <li>Retain the expected experimental columns A–E.</li>
  <li>Exclude column F from the standardized frequency-sweep dataset.</li>
  <li>Preserve column F and its formulas in the original workbook.</li>
  <li>Document the exclusion and its justification in the quality-control log.</li>
</ul>

<h3>Observation 2: Zero storage-modulus value</h3>

<p>
<strong>Original file:</strong>
<code>Colageno_bov_0.8_1_frecuencia.xlsx</code><br>
<strong>Worksheet:</strong> <code>Hoja1</code><br>
<strong>Cell:</strong> <code>C6</code>
</p>

<h4>Verified measurement</h4>

<table>
  <thead>
    <tr>
      <th>Parameter</th>
      <th>Verified value</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Measurement point</td>
      <td>1</td>
    </tr>
    <tr>
      <td>Angular frequency, ω</td>
      <td>0.628 rad/s</td>
    </tr>
    <tr>
      <td>Equivalent frequency, f</td>
      <td>Approximately 0.100 Hz</td>
    </tr>
    <tr>
      <td>Storage modulus, G′</td>
      <td>0 Pa</td>
    </tr>
    <tr>
      <td>Loss modulus, G″</td>
      <td>1.590 Pa</td>
    </tr>
    <tr>
      <td>Temperature</td>
      <td>37 °C</td>
    </tr>
    <tr>
      <td>Excel cell type</td>
      <td>Numeric</td>
    </tr>
  </tbody>
</table>

<h4>Evidence</h4>

<ul>
  <li>Cell <code>C6</code> contains a directly stored numeric value of <strong>0</strong>.</li>
  <li>The cell is not blank.</li>
  <li>The value is not produced by an Excel formula.</li>
  <li>At the next measurement point, G′ increases to <strong>0.986 Pa</strong> at 0.791 rad/s.</li>
  <li>The zero is therefore an isolated observation at the lowest measured angular frequency.</li>
</ul>

<h4>Interpretation</h4>

<ul>
  <li>The value may reflect the measured low-frequency response, instrument resolution, rounding, or another measurement-related condition.</li>
  <li>The workbook does not provide enough evidence to determine the exact cause.</li>
  <li>The zero must not be treated automatically as a missing value or data-entry error.</li>
</ul>

<h4>Data-handling decision</h4>

<ul>
  <li>Preserve the original value of G′ = 0 Pa.</li>
  <li>Retain the complete measurement row.</li>
  <li>Add a quality-control flag identifying the zero storage-modulus observation.</li>
  <li>Do not replace, impute, or remove the value automatically.</li>
  <li>
    Record the derived loss tangent as <code>NaN</code> or “not calculated”
    because <code>tan δ = G″/G′</code> is undefined when <code>G′ = 0</code>.
  </li>
</ul>

<h3>Overall conclusion</h3>

<ul>
  <li>
    <code>Colageno_bov_0.8_2_frecuencia.xlsx</code> contains an undocumented
    formula column that will be excluded from the standardized dataset.
  </li>
  <li>
    <code>Colageno_bov_0.8_1_frecuencia.xlsx</code> contains a confirmed
    numeric zero for G′ that will be retained and flagged.
  </li>
  <li>All processing decisions will be documented and reproducible.</li>
  <li>The original Excel workbooks will remain unchanged.</li>
</ul>

</details>

In [ ]:
# Step 5H — Exact character inspection of flagged unit labels

import unicodedata
import pandas as pd
from openpyxl import load_workbook


# These helper functions were created in Step 5G
required_helpers = ["locate_workbook", "find_sheet_and_header"]

missing_helpers = [
    name for name in required_helpers
    if name not in globals()
]

if missing_helpers:
    raise RuntimeError(
        "Please run the Step 5G inspection cell first. "
        f"Missing helper functions: {missing_helpers}"
    )


# Each entry specifies:
# original filename, variable to inspect, and expected canonical unit
unit_checks = [
    (
        "Colageno_bov_1.5_1_frecuencia.xlsx",
        "Temperature",
        "[°C]"
    ),
    (
        "Colageno_bov_1.5_1_tiempo.xlsx",
        "Complex Viscosity",
        "[Pa·s]"
    ),
    (
        "Colageno_bov_1.5_1_tiempo.xlsx",
        "Temperature",
        "[°C]"
    ),
    (
        "Colageno_bov_2.3_2_frecuencia.xlsx",
        "Temperature",
        "[°C]"
    ),
    (
        "Colageno_bov_2.3_2_tiempo.xlsx",
        "Complex Viscosity",
        "[Pa·s]"
    ),
    (
        "Colageno_bov_2.3_2_tiempo.xlsx",
        "Temperature",
        "[°C]"
    )
]


def describe_characters(value):
    """Return the Unicode identity of every character in a unit label."""
    if value is None:
        return "EMPTY CELL"

    text = str(value)

    return "; ".join(
        (
            f"{character!r} = U+{ord(character):04X} "
            f"({unicodedata.name(character, 'UNKNOWN CHARACTER')})"
        )
        for character in text
    )


unit_records = []

for filename, variable_name, expected_unit in unit_checks:
    workbook_path = locate_workbook(filename)

    workbook = load_workbook(
        workbook_path,
        read_only=True,
        data_only=False
    )

    worksheet, header_row, variable_column = find_sheet_and_header(
        workbook,
        variable_name
    )

    unit_cell = worksheet.cell(
        row=header_row + 1,
        column=variable_column
    )

    observed_unit = unit_cell.value

    unit_records.append({
        "original_filename": filename,
        "worksheet": worksheet.title,
        "variable": variable_name,
        "unit_cell": unit_cell.coordinate,
        "observed_unit": observed_unit,
        "observed_unit_repr": repr(observed_unit),
        "expected_unit": expected_unit,
        "exact_match": observed_unit == expected_unit,
        "observed_unicode_characters":
            describe_characters(observed_unit)
    })

    workbook.close()


unit_character_audit = pd.DataFrame(unit_records)

print("=" * 72)
print("EXACT CHARACTER INSPECTION OF FLAGGED UNIT LABELS")
print("=" * 72)
print(f"Unit labels inspected: {len(unit_character_audit)}")
print(
    "Exact matches:",
    int(unit_character_audit["exact_match"].sum())
)
print(
    "Non-matches:",
    int((~unit_character_audit["exact_match"]).sum())
)

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_columns", None,
    "display.max_rows", None
):
    display(unit_character_audit)

## Unit-label character-encoding audit

### Purpose

The six unit labels that failed the workbook-schema check were examined at the Unicode-character level. This check determined whether the differences represented genuine unit changes or character-encoding problems.

> **Note:** `Hoja1` is the worksheet-tab name inside each original Excel workbook. It is equivalent to `Sheet1` in an English-language workbook.

### Audit summary

| Audit measure | Result |
|---|---:|
| Unit labels inspected | 6 |
| Exact matches to the expected schema | 0 |
| Non-matching labels | 6 |
| Original workbooks affected | 4 |
| Distinct encoding patterns identified | 2 |

### Affected workbook locations

| Original workbook | Worksheet | Cell | Variable | Observed label | Expected label |
|---|---|---:|---|---|---|
| `Colageno_bov_1.5_1_frecuencia.xlsx` | `Hoja1` | `E5` | Temperature | `[ｰC]` | `[°C]` |
| `Colageno_bov_1.5_1_tiempo.xlsx` | `Hoja1` | `F5` | Complex viscosity | `[Paｷs]` | `[Pa·s]` |
| `Colageno_bov_1.5_1_tiempo.xlsx` | `Hoja1` | `H5` | Temperature | `[ｰC]` | `[°C]` |
| `Colageno_bov_2.3_2_frecuencia.xlsx` | `Hoja1` | `E5` | Temperature | `[ｰC]` | `[°C]` |
| `Colageno_bov_2.3_2_tiempo.xlsx` | `Hoja1` | `F5` | Complex viscosity | `[Paｷs]` | `[Pa·s]` |
| `Colageno_bov_2.3_2_tiempo.xlsx` | `Hoja1` | `H5` | Temperature | `[ｰC]` | `[°C]` |

### Exact character differences

| Variable | Observed character | Observed Unicode identity | Required character | Required Unicode identity |
|---|---|---|---|---|
| Temperature | `ｰ` | `U+FF70` — Halfwidth Katakana-Hiragana Prolonged Sound Mark | `°` | `U+00B0` — Degree Sign |
| Complex viscosity | `ｷ` | `U+FF77` — Halfwidth Katakana Letter Ki | `·` | `U+00B7` — Middle Dot |

### Interpretation

- The temperature label `[ｰC]` contains the Japanese halfwidth character `ｰ`, not a degree symbol.
- The complex-viscosity label `[Paｷs]` contains the Japanese halfwidth character `ｷ`, not a middle dot.
- The same two character substitutions occur consistently across the affected workbooks.
- The discrepancies are consistent with character-encoding or instrument-export artefacts.
- The variable headings and numerical measurement values remain present.
- There is no evidence that the underlying temperature or viscosity values use different physical units.
- Standardising these labels does **not** require numerical unit conversion.

### Canonical unit-label mapping

| Variable | Original label | Canonical label | Processing rule |
|---|---|---|---|
| Temperature | `[ｰC]` | `[°C]` | Replace the non-standard character only |
| Complex viscosity | `[Paｷs]` | `[Pa·s]` | Replace the non-standard character only |

### Data-handling decision

- Preserve all original Excel workbooks without modification.
- Preserve the original unit label as part of the source provenance.
- Apply the canonical unit label only when constructing the processed dataset.
- Do not change any numerical measurement values during label standardisation.
- Record the original label, canonical label, workbook name, worksheet, cell location, and mapping decision in the quality-control documentation.
- Validate every mapped label against the expected schema before saving the processed data.

### Conclusion

All six flagged unit-label discrepancies are explained by two repeatable Unicode-character substitutions. They are label-encoding issues rather than missing measurements or genuine differences in physical units. The labels can therefore be standardised reproducibly in the processed-data workflow while the original workbooks remain unchanged.

In [ ]:
# Step 5I — Validate canonical unit-label mappings in memory

import pandas as pd
from IPython.display import display

if "unit_character_audit" not in globals():
    raise RuntimeError(
        "unit_character_audit was not found. "
        "Run the exact-character inspection cell first."
    )

# Exact Unicode mappings
UNIT_LABEL_MAP = {
    "[\uFF70C]": "[\u00B0C]",       # [ｰC]   -> [°C]
    "[Pa\uFF77s]": "[Pa\u00B7s]",   # [Paｷs] -> [Pa·s]
}

# Work on a copy; the audit table and Excel workbooks remain unchanged
unit_mapping_validation = unit_character_audit.copy()

unit_mapping_validation["standardized_unit"] = (
    unit_mapping_validation["observed_unit"].map(UNIT_LABEL_MAP)
)

unit_mapping_validation["mapping_found"] = (
    unit_mapping_validation["standardized_unit"].notna()
)

unit_mapping_validation["mapping_success"] = (
    unit_mapping_validation["mapping_found"]
    & (
        unit_mapping_validation["standardized_unit"]
        == unit_mapping_validation["expected_unit"]
    )
)

labels_tested = len(unit_mapping_validation)
mappings_found = int(unit_mapping_validation["mapping_found"].sum())
successful_mappings = int(unit_mapping_validation["mapping_success"].sum())
unresolved_labels = labels_tested - successful_mappings

print("=" * 72)
print("VALIDATION OF CANONICAL UNIT-LABEL MAPPINGS")
print("=" * 72)
print(f"Labels tested:       {labels_tested}")
print(f"Mappings found:      {mappings_found}")
print(f"Successful mappings: {successful_mappings}")
print(f"Unresolved labels:   {unresolved_labels}")

display_columns = [
    "original_filename",
    "worksheet",
    "variable",
    "unit_cell",
    "observed_unit",
    "standardized_unit",
    "expected_unit",
    "mapping_success",
]

display(unit_mapping_validation[display_columns])

if not unit_mapping_validation["mapping_success"].all():
    raise ValueError(
        "At least one unit label was not mapped to the expected canonical form."
    )

print("\nValidation passed: all flagged unit labels were mapped successfully.")
print("No original Excel workbook or numerical measurement was modified.")

<details>
<summary><strong>Canonical unit-label mapping validation</strong></summary>

### Validation objective

The proposed unit-label mappings were tested against all six discrepancies identified during the exact Unicode-character audit. The purpose was to confirm that each non-standard source label could be mapped reproducibly to its expected canonical form before applying the mappings to the processed dataset.

### Validation summary

| Validation metric | Result |
|---|---:|
| Unit labels tested | 6 |
| Mappings found | 6 |
| Successful mappings | 6 |
| Unresolved labels | 0 |
| Overall validation status | **Passed** |

### Confirmed mapping rules

| Variable | Observed source label | Canonical label | Character correction |
|---|---|---|---|
| Temperature | `[ｰC]` | `[°C]` | `U+FF70` replaced with degree symbol `U+00B0` |
| Complex viscosity | `[Paｷs]` | `[Pa·s]` | `U+FF77` replaced with middle dot `U+00B7` |

These mappings correct only the non-standard Unicode characters. They do not represent physical unit conversions.

### Affected source locations

| Original workbook | Worksheet | Cell | Variable | Observed label | Canonical label | Validation |
|---|---|---:|---|---|---|---|
| `Colageno_bov_1.5_1_frecuencia.xlsx` | `Hoja1` | `E5` | Temperature | `[ｰC]` | `[°C]` | Passed |
| `Colageno_bov_1.5_1_tiempo.xlsx` | `Hoja1` | `F5` | Complex viscosity | `[Paｷs]` | `[Pa·s]` | Passed |
| `Colageno_bov_1.5_1_tiempo.xlsx` | `Hoja1` | `H5` | Temperature | `[ｰC]` | `[°C]` | Passed |
| `Colageno_bov_2.3_2_frecuencia.xlsx` | `Hoja1` | `E5` | Temperature | `[ｰC]` | `[°C]` | Passed |
| `Colageno_bov_2.3_2_tiempo.xlsx` | `Hoja1` | `F5` | Complex viscosity | `[Paｷs]` | `[Pa·s]` | Passed |
| `Colageno_bov_2.3_2_tiempo.xlsx` | `Hoja1` | `H5` | Temperature | `[ｰC]` | `[°C]` | Passed |

### Interpretation

- All six flagged labels were matched to an explicitly defined canonical label.
- Every standardised label agrees exactly with the corresponding expected unit in the workbook schema.
- The discrepancies are attributable to two repeatable Unicode-character substitutions.
- No evidence of missing measurements or genuine differences in physical units was found.
- No numerical rheological measurements were altered during validation.

### Data-handling decision

- Preserve all original Excel workbooks without modification.
- Preserve the original unit labels as source-provenance fields.
- Apply the canonical labels only when constructing the processed dataset.
- Record the workbook name, worksheet, cell location, original label, canonical label, and mapping result in the quality-control documentation.
- Reject or flag any future unit label that is not covered by an approved mapping rule.

### Conclusion

The canonical unit-label mapping procedure passed validation. All six known discrepancies can be standardised reproducibly in the processed-data workflow without changing the numerical measurements or the original Excel workbooks.

</details>

In [ ]:
# Step 5J — Export the validated unit-label mapping audit

from pathlib import Path
import pandas as pd

if "unit_mapping_validation" not in globals():
    raise RuntimeError(
        "unit_mapping_validation was not found. "
        "Run Step 5I before exporting the audit table."
    )

if not unit_mapping_validation["mapping_success"].all():
    raise ValueError(
        "The audit table was not exported because one or more mappings failed."
    )

# Identify the project root
current_directory = Path.cwd().resolve()
project_root = (
    current_directory.parent
    if current_directory.name.lower() in {"notebook", "notebooks"}
    else current_directory
)

# Create a dedicated quality-control directory
quality_control_directory = project_root / "data" / "quality_control"
quality_control_directory.mkdir(parents=True, exist_ok=True)

output_file = (
    quality_control_directory
    / "unit_label_mapping_audit.csv"
)

# Document the exact Unicode-character substitutions
unicode_mapping_rules = {
    "[\uFF70C]": "U+FF70 -> U+00B0",
    "[Pa\uFF77s]": "U+FF77 -> U+00B7",
}

unit_label_mapping_audit = unit_mapping_validation[
    [
        "original_filename",
        "worksheet",
        "variable",
        "unit_cell",
        "observed_unit",
        "standardized_unit",
        "expected_unit",
        "mapping_found",
        "mapping_success",
    ]
].copy()

unit_label_mapping_audit["unicode_mapping"] = (
    unit_label_mapping_audit["observed_unit"].map(
        unicode_mapping_rules
    )
)

unit_label_mapping_audit["processing_decision"] = (
    "Standardise the unit label in the processed dataset only"
)

unit_label_mapping_audit["source_workbook_modified"] = False

# Arrange the exported columns clearly
unit_label_mapping_audit = unit_label_mapping_audit[
    [
        "original_filename",
        "worksheet",
        "unit_cell",
        "variable",
        "observed_unit",
        "standardized_unit",
        "expected_unit",
        "unicode_mapping",
        "mapping_found",
        "mapping_success",
        "processing_decision",
        "source_workbook_modified",
    ]
]

# UTF-8 with BOM preserves the special characters when opened in Excel
unit_label_mapping_audit.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig",
)

# Reload and validate the saved file
saved_audit = pd.read_csv(
    output_file,
    encoding="utf-8-sig",
)

if len(saved_audit) != len(unit_label_mapping_audit):
    raise ValueError(
        "The number of saved audit records does not match the source table."
    )

if not saved_audit["mapping_success"].all():
    raise ValueError(
        "At least one saved mapping does not have a successful status."
    )

print("=" * 72)
print("UNIT-LABEL MAPPING AUDIT EXPORT")
print("=" * 72)
print(f"Records exported:       {len(saved_audit)}")
print(f"Successful mappings:    {int(saved_audit['mapping_success'].sum())}")
print("Source workbooks edited: No")
print(f"Saved file:             {output_file.relative_to(project_root).as_posix()}")

display(saved_audit)

## Consolidated data-quality issue log

<details>
<summary><strong>Purpose, scope and processing rules</strong></summary>

### Objective

This step consolidates all confirmed workbook-level, schema-level and measurement-level findings into a single reproducible quality-control issue log.

The issue log provides a structured audit trail linking each finding to:

- its original Excel workbook;
- worksheet and cell location;
- affected rheological variable;
- observed condition;
- expected or canonical condition;
- scientific interpretation;
- processing decision; and
- resolution status.

The original Excel workbooks will remain unchanged.

### Findings included

A total of **eight confirmed quality-control findings** will be documented.

| Issue category | Number of records | Affected source |
|---|---:|---|
| Unexpected schema column | 1 | `Colageno_bov_0.8_2_frecuencia.xlsx` |
| Zero storage-modulus observation | 1 | `Colageno_bov_0.8_1_frecuencia.xlsx` |
| Non-standard Unicode unit labels | 6 | Four frequency- and time-sweep workbooks |
| **Total** | **8** | — |

### 1. Unexpected schema column

In `Colageno_bov_0.8_2_frecuencia.xlsx`, worksheet `Hoja1`, column `F` is not part of the expected frequency-sweep schema.

- Cells `F6:F15` are empty.
- Cells `F16:F26` contain formulas of the form `=D[row]+1`.
- The column has no heading or unit label.
- Its scientific meaning cannot be verified from the workbook.

**Processing decision:** Preserve the column in the original workbook but exclude it from the standardised measurement dataset.

### 2. Zero storage-modulus observation

In `Colageno_bov_0.8_1_frecuencia.xlsx`, worksheet `Hoja1`, cell `C6` contains a numeric storage-modulus value of \(G' = 0\ \mathrm{Pa}\).

The associated measurement conditions are:

| Measurement | Value |
|---|---:|
| Angular frequency | 0.628 rad/s |
| Storage modulus, \(G'\) | 0 Pa |
| Loss modulus, \(G''\) | 1.590 Pa |
| Temperature | 37 °C |

The workbook does not provide sufficient evidence to determine whether the zero reflects instrument resolution, rounding, an export condition or a genuine low-frequency response.

**Processing decision:** Retain the original value, add a quality-control flag and record the derived loss tangent as missing because

\[
\tan\delta = \frac{G''}{G'}
\]

is undefined when \(G' = 0\).

### 3. Non-standard Unicode unit labels

Six unit labels contain visually similar but incorrect Unicode characters:

| Variable | Observed label | Canonical label | Character mapping |
|---|---|---|---|
| Temperature | `[ｰC]` | `[°C]` | `U+FF70 → U+00B0` |
| Complex viscosity | `[Paｷs]` | `[Pa·s]` | `U+FF77 → U+00B7` |

All six mappings were tested successfully against the expected workbook schemas.

**Processing decision:** Preserve each original label for provenance and apply the canonical label only when constructing the processed dataset.

### Quality-control principles

The consolidated issue log follows these rules:

- Raw Excel workbooks are treated as immutable source records.
- No numerical rheological measurement is silently changed or deleted.
- Metadata corrections are applied only through documented mapping rules.
- Uncertain measurements are retained with explicit quality-control flags.
- Undocumented columns are excluded only from the canonical dataset, not removed from the source workbook.
- Every processing decision remains traceable to its original workbook, worksheet and cell location.

### Expected output

The following machine-readable audit file will be created:

```text
data/quality_control/data_quality_issue_log.csv
```

</details>

In [ ]:
# Step 5K — Create the consolidated data-quality issue log

from pathlib import Path
import pandas as pd

# Locate the project directory
current_directory = Path.cwd().resolve()
project_root = (
    current_directory.parent
    if current_directory.name.lower() in {"notebook", "notebooks"}
    else current_directory
)

quality_control_directory = project_root / "data" / "quality_control"
unit_audit_file = (
    quality_control_directory
    / "unit_label_mapping_audit.csv"
)

if not unit_audit_file.exists():
    raise FileNotFoundError(
        f"Required audit file was not found: {unit_audit_file}"
    )

# Reload the validated unit-label audit
unit_audit = pd.read_csv(
    unit_audit_file,
    encoding="utf-8-sig",
)

mapping_success = (
    unit_audit["mapping_success"]
    .astype(str)
    .str.lower()
    .eq("true")
)

if not mapping_success.all():
    raise ValueError(
        "The consolidated issue log cannot be created because "
        "at least one unit-label mapping is unresolved."
    )

# -------------------------------------------------------------------
# 1. Non-unit issues identified during workbook inspection
# -------------------------------------------------------------------

issue_records = [
    {
        "issue_id": "SCHEMA-001",
        "original_filename": "Colageno_bov_0.8_2_frecuencia.xlsx",
        "worksheet": "Hoja1",
        "cell_or_range": "F6:F26",
        "variable": "Unnamed column F",
        "issue_category": "Unexpected schema column",
        "observed_value": (
            "F6:F15 are empty; F16:F26 contain formulas "
            "of the form =D[row]+1"
        ),
        "expected_or_canonical_value": (
            "No sixth column is defined in the "
            "frequency-sweep schema"
        ),
        "scientific_interpretation": (
            "The column is an undocumented derived field whose "
            "scientific meaning cannot be verified from the workbook."
        ),
        "processing_decision": (
            "Exclude column F from the standardised dataset while "
            "preserving it in the original workbook."
        ),
        "resolution_status": "Resolved by processing rule",
        "source_workbook_modified": False,
    },
    {
        "issue_id": "VALUE-001",
        "original_filename": "Colageno_bov_0.8_1_frecuencia.xlsx",
        "worksheet": "Hoja1",
        "cell_or_range": "C6",
        "variable": "Storage Modulus (G′)",
        "issue_category": "Zero rheological value",
        "observed_value": (
            "G′ = 0 Pa at 0.628 rad/s; "
            "G″ = 1.590 Pa; temperature = 37 °C"
        ),
        "expected_or_canonical_value": (
            "No replacement value imposed"
        ),
        "scientific_interpretation": (
            "An isolated numeric zero at the lowest measured angular "
            "frequency. Its cause cannot be determined from the "
            "available workbook evidence."
        ),
        "processing_decision": (
            "Retain G′ = 0 Pa, flag the observation, and record "
            "tan δ as missing because division by zero is undefined."
        ),
        "resolution_status": "Retained with quality-control flag",
        "source_workbook_modified": False,
    },
]

# -------------------------------------------------------------------
# 2. Add the six validated unit-label issues
# -------------------------------------------------------------------

for record_number, row in unit_audit.reset_index(drop=True).iterrows():
    issue_records.append(
        {
            "issue_id": f"UNIT-{record_number + 1:03d}",
            "original_filename": row["original_filename"],
            "worksheet": row["worksheet"],
            "cell_or_range": row["unit_cell"],
            "variable": row["variable"],
            "issue_category": "Non-standard Unicode unit label",
            "observed_value": row["observed_unit"],
            "expected_or_canonical_value": row["standardized_unit"],
            "scientific_interpretation": (
                "The discrepancy is an encoding or export-character "
                "substitution, not a difference in physical units."
            ),
            "processing_decision": (
                "Standardise the unit label in the processed dataset "
                "while preserving the original source label."
            ),
            "resolution_status": "Mapped and validated",
            "source_workbook_modified": False,
        }
    )

# Construct the consolidated issue log
data_quality_issue_log = pd.DataFrame(issue_records)

expected_record_count = 2 + len(unit_audit)

if len(data_quality_issue_log) != expected_record_count:
    raise ValueError(
        "The consolidated issue-log record count is incorrect."
    )

if data_quality_issue_log["issue_id"].duplicated().any():
    raise ValueError(
        "Duplicate issue identifiers were detected."
    )

# Save the issue log
issue_log_file = (
    quality_control_directory
    / "data_quality_issue_log.csv"
)

data_quality_issue_log.to_csv(
    issue_log_file,
    index=False,
    encoding="utf-8-sig",
)

# Reload the exported file for verification
saved_issue_log = pd.read_csv(
    issue_log_file,
    encoding="utf-8-sig",
)

if len(saved_issue_log) != expected_record_count:
    raise ValueError(
        "The saved issue-log record count does not match "
        "the expected count."
    )

workbook_modified = (
    saved_issue_log["source_workbook_modified"]
    .astype(str)
    .str.lower()
    .eq("true")
)

if workbook_modified.any():
    raise ValueError(
        "The audit log incorrectly indicates that a source "
        "workbook was modified."
    )

print("=" * 72)
print("CONSOLIDATED DATA-QUALITY ISSUE LOG")
print("=" * 72)
print(f"Issues documented:       {len(saved_issue_log)}")
print(f"Schema issues:           {(saved_issue_log['issue_id'].str.startswith('SCHEMA')).sum()}")
print(f"Value observations:      {(saved_issue_log['issue_id'].str.startswith('VALUE')).sum()}")
print(f"Unit-label issues:       {(saved_issue_log['issue_id'].str.startswith('UNIT')).sum()}")
print("Source workbooks edited: No")
print(f"Saved file:              {issue_log_file.relative_to(project_root).as_posix()}")

display(
    saved_issue_log[
        [
            "issue_id",
            "original_filename",
            "worksheet",
            "cell_or_range",
            "variable",
            "issue_category",
            "resolution_status",
        ]
    ]
)

<details>
<summary><strong>Consolidated data-quality issue-log results</strong></summary>

### Audit outcome

The consolidated quality-control audit documented **eight findings** across the rheology workbooks. Every finding now has a defined processing decision and traceable resolution status.

| Audit metric                    | Result |
| ------------------------------- | -----: |
| Total issues documented         |      8 |
| Unexpected schema-column issues |      1 |
| Zero-value observations         |      1 |
| Non-standard unit-label issues  |      6 |
| Source workbooks modified       | **No** |

### Resolution summary

| Issue identifier      | Finding                                                                      | Processing decision                                                                                         | Status                             |
| --------------------- | ---------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------- | ---------------------------------- |
| `SCHEMA-001`          | Undocumented column `F` in `Colageno_bov_0.8_2_frecuencia.xlsx`              | Exclude the column from the standardised dataset while preserving the original workbook                     | Resolved by processing rule        |
| `VALUE-001`           | \(G' = 0\ \mathrm{Pa}\) in cell `C6` of `Colageno_bov_0.8_1_frecuencia.xlsx` | Retain the measurement, add a quality-control flag and do not calculate \(\tan\delta\) for this observation | Retained with quality-control flag |
| `UNIT-001`–`UNIT-006` | Six non-standard Unicode unit labels across four workbooks                   | Replace only the non-standard label characters when constructing the processed dataset                      | Mapped and validated               |

### Schema issue

`SCHEMA-001` records the unexpected sixth column in `Colageno_bov_0.8_2_frecuencia.xlsx`, worksheet `Hoja1`, range `F6:F26`.

The column is not defined in the working frequency-sweep schema inferred from the consistent workbook structure and does not contain a heading or unit. Because its scientific meaning cannot be verified, it will not be included in the standardised dataset. The source workbook and its formulas will remain unchanged.

### Zero storage-modulus observation

`VALUE-001` records the numeric value \(G' = 0\ \mathrm{Pa}\) in `Colageno_bov_0.8_1_frecuencia.xlsx`, worksheet `Hoja1`, cell `C6`.

The value will be retained because there is insufficient evidence to classify it as missing or erroneous. It will receive an explicit quality-control flag. The derived loss tangent will be recorded as missing for this row because

$$
\tan\delta = \frac{G''}{G'}
$$

is undefined when \(G' = 0\).

### Unit-label issues

`UNIT-001` to `UNIT-006` document six non-standard Unicode unit labels. All six labels were mapped successfully to the expected canonical forms:

* `[ｰC]` was standardised to `[°C]`.
* `[Paｷs]` was standardised to `[Pa·s]`.

These are metadata corrections only. They do not represent physical unit conversions and do not alter any numerical measurements.

### Interpretation of resolution status

A documented resolution status means that a reproducible **data-handling decision** has been established. It does not necessarily mean that the underlying experimental cause is known.

In particular, the cause of the zero storage-modulus observation remains scientifically uncertain. The observation is therefore retained and flagged rather than corrected, replaced or removed.

### Provenance and reproducibility

The consolidated issue log preserves the following information for every finding:

* issue identifier;
* original workbook name;
* worksheet and cell location;
* affected variable;
* issue category;
* scientific interpretation;
* processing decision; and
* resolution status.

No original Excel workbook or numerical rheological measurement was modified during this process.

### Output artifact

The completed audit log was exported to:

`data/quality_control/data_quality_issue_log.csv`

This file will serve as the central quality-control reference during construction and validation of the standardised rheology datasets.

### Conclusion

The workbook-audit stage is complete. All eight confirmed findings are now documented through explicit, reproducible and traceable processing rules. The dataset can proceed to canonical schema definition and controlled extraction while the original experimental records remain unchanged.

</details>
